# 개별종목 조합J — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합J 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합J의 피처 값만 지정합니다.
import json

COMBINATION = 'J'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합J 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4988,0.5012,-0.0024,0.3234,0.3655,0.0838,0.3782,0.1186,0.2217
1,2,balanced,980,20150123,20150421,0.3939,0.3978,-0.0039,0.3597,0.3685,0.0631,0.3755,0.2256,0.3076
2,3,balanced,1210,20151228,20160328,0.3601,0.3762,-0.0161,0.3599,0.3606,0.0437,0.3752,0.3720,0.3639
3,4,balanced,1439,20161202,20170228,0.4501,0.4617,-0.0116,0.3604,0.3770,0.0851,0.3994,0.1962,0.2972
4,5,balanced,1669,20171113,20180207,0.4213,0.3901,0.0312,0.3867,0.3979,0.1080,0.3933,0.2968,0.3602
5,6,balanced,1899,20181024,20190118,0.4114,0.3725,0.0389,0.4104,0.4200,0.1328,0.4229,0.5284,0.4438
6,7,balanced,2129,20190930,20191224,0.4638,0.4781,-0.0143,0.3517,0.3738,0.0865,0.4000,0.1879,0.2906
7,8,balanced,2359,20200902,20201130,0.3946,0.3476,0.0469,0.3900,0.3938,0.0924,0.4007,0.4097,0.3979
8,9,balanced,2589,20210806,20211105,0.3961,0.3916,0.0045,0.3858,0.3920,0.0858,0.3996,0.3369,0.3710
9,10,balanced,2818,20220714,20221012,0.3509,0.3454,0.0055,0.3509,0.3534,0.0318,0.3637,0.2942,0.3297


,OOS 폴드 평균
accuracy,0.4115
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0146
macro_f1,0.3726
balanced_accuracy,0.3829
mcc,0.0839
pr_auc_macro_ovr,0.3922
down_recall,0.3060
core_harmonic_mean,0.3455


재실행 명령: python scripts/run_stock_model_experiment.py
